In [ ]:
!pip install google-api-python-client youtube-comment-downloader pandas

In [ ]:
from googleapiclient.discovery import build
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
import re

# -----------------------
# 1. API 설정
# -----------------------
API_KEY = "api_key"  # TODO: replace with your YouTube Data API key (do not commit real keys)

youtube = build("youtube", "v3", developerKey=API_KEY)

# -----------------------
# 2. 검색 키워드
# -----------------------
keywords = [
    "층간소음",
    "매트시공",
    "소음",
    "시끄러움",
    "시끄럽다",
    "발망치",
    "발소리",
    "쿵쿵",
    "의자 끄는 소리",
    "화장실",
    "강아지 짖는 소리",
    "문 쾅",
    "문콕",
    "새벽 드라이기",
    "청소기 소리",
    "차 소리",
    "경적",
    "덜컹",
    "아랫집 항의",
    "층간소음 스트레스",
    "층간소음 해결",
    "층간소음 신고",
    "층간소음 경찰",
    "층간소음 관리사무소",
    "층간소음 보복",
    "층간소음 참교육",
    "층간소음 발망치",
    "윗집 발망치",
    "윗집 소음",
    "새벽 층간소음",
    "아이 뛰는 소리 층간소음",
    "아파트 층간소음",
    "층간소음 방음매트",
    "층간소음 측정",
    "층간소음 민원",
    "층간소음 법",
    "층간소음 사건",
    "층간소음 이웃사이센터"
]

videos = []

# -----------------------
# 3. 유튜브 영상 검색
# -----------------------

seen_ids = set()
for keyword in keywords:

    request = youtube.search().list(
        q=keyword,
        part="id,snippet",
        maxResults=25,
        type="video"
    )

    response = request.execute()

    for item in response.get("items",[]):
        video_id = item.get("id", {}).get("videoId")
        if video_id and video_id not in seen_ids:
            title = item["snippet"]["title"]
            videos.append((video_id, title))
            seen_ids.add(video_id)

print("수집된 영상 수:", len(videos))


# -----------------------
# 4. 댓글 수집
# -----------------------
downloader = YoutubeCommentDownloader()

data = []

for video_id, title in videos:

    url = f"https://www.youtube.com/watch?v={video_id}"

    try:
        comments = downloader.get_comments_from_url(url)

        count = 0

        # -----------------------
        # 4. 댓글 수집 부분 수정
        # -----------------------
        for comment in comments:
            contents = comment["text"]
            date = comment["time"] # 예: "1년 전" 또는 "1 year ago"
            author = comment["author"]

            
            # 1. 한국어 '년' 또는 영어 'year'가 포함되어 있으면 제외 (1년 이상 된 데이터)
            if ("년" in date) or ("year" in date):
                match = re.search(r'\d+', date)
                if match:
                    years = int(match.group())

                    # 4년 이상 된 댓글은 건너뛰기 (3년 전까지만 수집)
                    if years > 3:
                        continue
        
            
        
            data.append({
                "title": title,
                "date": date,
                "author" : author,
                "contents": contents
                
            })
        
            count += 1
            if count == 200:
                break

    except:
        continue


# -----------------------
# 5. DataFrame 생성
# -----------------------
df = pd.DataFrame(data)

print(df.head())
print("총 댓글 수:", len(df))


# -----------------------
# 6. CSV 저장
# -----------------------
df.to_csv("층간소음_youtube.csv", index=False)

In [ ]:
df.to_excel('층간소음_youtube.xlsx', index=False)

In [ ]:
!pip install openpyxl